# Preparation

In [2]:
import json
import pandas as pd

from dotenv import load_dotenv
from embedder import Embedder
from evaluation_utils import calc_total_price, llm_structured_retry
from gitsource import chunk_documents, GithubRepositoryDataReader
from minsearch import Index, VectorSearch
from openai import OpenAI
from pydantic import BaseModel
from tqdm import tqdm

2026-07-11 10:17:08.111732976 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
len(documents)

72

In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [6]:
def generate_ground_truth_for_page(page: dict, client) -> tuple[list[dict], object]:
    """Generate 5 labeled questions for one lesson page.

    Returns the question records AND the raw usage object separately —
    keeping them apart means a downstream cost-calculation bug can never
    corrupt the ground-truth records themselves, and vice versa. Coupling
    unrelated concerns (labeled data + token accounting) into one dict
    is the kind of thing that's fine until you need to touch one without
    the other.
    """
    # Structured-output prompts want a clean, unambiguous payload —
    # JSON avoids the LLM guessing where the filename ends and content begins,
    # which free-form string concatenation can't guarantee.
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"],
    })
    class Questions(BaseModel):
        questions: list[str]
    
    parsed, usage = llm_structured_retry(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
    )

    records = [
        {"question": q, "filename": page["filename"]}
        for q in parsed.questions
    ]

    return records, usage

In [7]:
def build_ground_truth(documents: list[dict], client) -> tuple[list[dict], list]:
    """Generate ground truth across all lesson pages.

    Sequential on purpose for now — see note below on concurrency.
    """
    all_records = []
    all_usages = []

    for page in tqdm(documents):
        records, usage = generate_ground_truth_for_page(page, client)
        all_records.extend(records)
        all_usages.append(usage)

    return all_records, all_usages

In [8]:
load_dotenv()
openai_client = OpenAI()

In [9]:
ground_truth_records, usages = build_ground_truth(documents, openai_client)
# print(f"generated {len(ground_truth_records)} questions from {len(documents)} pages")
# print(f"total cost: ${calc_total_price(usages):.4f}")

100%|██████████| 72/72 [01:59<00:00,  1.66s/it]


# Question 1

In [10]:
target_filenames = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}

sample_pages = [doc for doc in documents if doc["filename"] in target_filenames]

# guard against a silent typo/path mismatch — if this doesn't print 3,
# something about how `filename` is stored in `documents` doesn't match
# what you typed above (leading slash, different separator, etc.)
print(f"matched {len(sample_pages)} / {len(target_filenames)} target pages")
assert len(sample_pages) == 3, "check filename formatting against documents"

sample_records, sample_usages = build_ground_truth(sample_pages, openai_client)

print(f"generated {len(sample_records)} questions from {len(sample_pages)} pages")
print(f"total cost: ${calc_total_price(sample_usages):.4f}")

matched 3 / 3 target pages


100%|██████████| 3/3 [00:04<00:00,  1.42s/it]

generated 15 questions from 3 pages
total cost: $0.0045


In [11]:
for page, usage in zip(sample_pages, sample_usages):
    print(f"{page['filename']}: {usage.input_tokens} input tokens")

01-agentic-rag/lessons/01-intro.md: 1021 input tokens
01-agentic-rag/lessons/02-environment.md: 1287 input tokens
01-agentic-rag/lessons/03-rag.md: 1754 input tokens


In [12]:
question1_answer = 1400

In [13]:
ground_truth = pd.DataFrame(ground_truth_records)
ground_truth.head()

,question,filename
0,What problem does retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG setup in pl...,01-agentic-rag/lessons/01-intro.md
2,What are the main limits of large language mod...,01-agentic-rag/lessons/01-intro.md
3,What kind of app is this module building as a ...,01-agentic-rag/lessons/01-intro.md
4,"How is the module split up, and what changes i...",01-agentic-rag/lessons/01-intro.md


In [14]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

## VectorSearch

In [15]:
vs = VectorSearch()
chunk_texts = [c["content"] for c in chunks]
embedding_instance = Embedder()
X = embedding_instance.encode_batch(chunk_texts)
vs.fit(X, payload=chunks)        # index your embedded chunks + their metadata

text_search_index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)
text_search_index.fit(chunks)

In [ ]:
def text_search(query: str, num_results: int = 5) -> list[dict]:
    """Lexical search over chunk content, boosted toward filename-scoped matches."""
    return text_search_index.search(
        query,
        num_results=num_results,
    )


def vector_search(query: str, num_results: int = 5) -> list[dict]:
    """Semantic search over chunk embeddings.

    encode_batch takes a list even for a single query — reusing it here
    instead of assuming a separate single-query method avoids depending
    on an API surface that might not exist on Embedder.
    """
    query_vector = Embedder().encode_batch([query])[0]
    return vs.search(query_vector, num_results=num_results)


def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query: str, k: int = 60, num_results: int = 5) -> list[dict]:
    """RRF fusion of lexical + vector results.

    Each branch pulls 10 candidates (wider than the final num_results)
    so RRF has enough overlap signal to work with — fusing two 5-result
    lists barely gives rank-fusion anything to do.
    """
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

# Question 2

In [17]:
q = ground_truth_records[0]["question"]
q

'What problem does retrieval-augmented generation solve for a chatbot or QA system?'

In [18]:
text_search(q)

[{'start': 0,
  'content': '# Retrieval Augmented Generation\n\nVideo: [RAG Workflows](https://youtu.be/FhGZV173xrk)\n\nAI Copilot solves the context problem for flow generation. But what about workflows that need to answer questions from your own data? That\'s where RAG comes in.\n\n> Note: Flows 1 and 2 use `{{ secret(\'GEMINI_API_KEY\') }}`. Flow 3 uses `{{ secret(\'OPENAI_API_KEY\') }}` and `{{ secret(\'TAVILY_API_KEY\') }}`. Make sure you\'ve completed the [setup instructions](03-setup.md) to configure the relevant secrets before running them.\n\n## What is RAG?\n\nRAG (Retrieval Augmented Generation) is a technique that retrieves relevant information from your data sources, augments the AI prompt with that context, and generates a response grounded in real data. This solves the hallucination problem by ensuring the AI has access to current, accurate information at query time.\n\nFor a deeper dive into RAG concepts, see [Module 1: Intro to RAG](../../01-agentic-rag/lessons/03-rag.

In [19]:
question2_answer = '01-agentic-rag/lessons/03-rag.md'

# Question 3

In [20]:
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [21]:
question3_answer = '01-agentic-rag/lessons/01-intro.md'

# Question 4

In [22]:
def compute_relevance(q, search_function):
    """Check whether the correct page (by filename) shows up in results.

    Only difference from the module's version: ground truth for lesson pages
    is labeled by `filename`, and search results are chunks carrying their
    own `filename` field (from `chunks` metadata) — no `document`/`id` here,
    since we're chunking pages rather than working with atomic FAQ entries.
    """
    target_filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == target_filename))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
    return relevance_total


def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt += 1
    return cnt / len(relevance)


def mean_reciprocal_rank(relevance: list[list[int]]) -> float:
    total_rr = 0.0
    for line in relevance:
        if 1 in line:
            total_rr += 1 / (line.index(1) + 1)
        # else: contributes 0, but still counts toward the denominator
    return total_rr / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mean_reciprocal_rank(relevance_total),
    }

In [25]:
evaluate(ground_truth_records, text_search)

100%|██████████| 360/360 [00:01<00:00, 211.30it/s]


{'hit_rate': 0.7666666666666667, 'mrr': 0.6114814814814815}

In [26]:
question4_answer = 0.76

# Question 5

In [27]:
question5_answer = evaluate(ground_truth_records, vector_search)["mrr"]

100%|██████████| 360/360 [01:34<00:00,  3.79it/s]


In [28]:
print(question5_answer)

0.5499537037037038


# Question 6

In [32]:
k_vals = [1, 50, 100, 200]

for k in k_vals:
    result = evaluate(
        ground_truth_records,
        lambda query, k=k: hybrid_search(query, k=k)
    )
    print(f"k = {k}: mrr = {result['mrr']:.4f}")

100%|██████████| 360/360 [01:29<00:00,  4.02it/s]


k = 1: mrr = 0.6719


100%|██████████| 360/360 [01:18<00:00,  4.56it/s]


k = 50: mrr = 0.6431


100%|██████████| 360/360 [01:14<00:00,  4.82it/s]


k = 100: mrr = 0.6431


100%|██████████| 360/360 [01:13<00:00,  4.91it/s]

k = 200: mrr = 0.6431


In [34]:
question6_answer = 50